# LeetCode 1605 — Find Valid Matrix Given Row and Column Sums bằng Python

Link đề: https://leetcode.com/problems/find-valid-matrix-given-row-and-column-sums/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Matrix, Flow Network

Skeleton LeetCode:

```python
class Solution:
    def restoreMatrix(self, rowSum: List[int], colSum: List[int]) -> List[List[int]]:
        ...
```

## 1. Mục tiêu học

1. Hiểu bài "**dựng một đáp án bất kỳ**" (nhiều đáp án đúng) và cách kiểm thử bằng **hàm kiểm tra hợp lệ** thay vì so khớp một ma trận cố định.
2. Viết baseline **quay lui** thử giá trị từng ô (chỉ cho input nhỏ) để thấy không gian tìm kiếm.
3. Nắm chiến lược **tham lam `min(rowSum[i], colSum[j])`**: đặt giá trị lớn nhất có thể vào một ô làm "cạn" ít nhất một hàng hoặc một cột.
4. Cài đặt bằng **duyệt từng ô** và bằng **hai con trỏ** (O(m + n) bước điền), cùng phiên bản theo gợi ý "chọn tổng nhỏ nhất" bằng heap.
5. Chứng minh vì sao tham lam luôn thành công khi `sum(rowSum) == sum(colSum)`.

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên **không âm** `rowSum` và `colSum`: `rowSum[i]` là tổng các phần tử hàng `i`, `colSum[j]` là tổng các phần tử cột `j` của một ma trận 2 chiều mà ta **không biết** các phần tử.

Hãy tìm **một ma trận bất kỳ** gồm các số nguyên **không âm**, kích thước `len(rowSum) × len(colSum)`, thỏa mãn các tổng hàng và tổng cột. Đề **đảm bảo** luôn tồn tại ít nhất một ma trận như vậy.

**Ví dụ 1:** `rowSum = [3,8]`, `colSum = [4,7]` → `[[3,0],[1,7]]`. Kiểm tra: hàng `3+0=3`, `1+7=8`; cột `3+1=4`, `0+7=7`. Một đáp án khác: `[[1,2],[3,5]]`.

**Ví dụ 2:** `rowSum = [5,7,10]`, `colSum = [8,6,8]` → ví dụ `[[0,5,0],[6,1,0],[2,0,8]]`.

**Ràng buộc:**
- `1 ≤ rowSum.length, colSum.length ≤ 500`
- `0 ≤ rowSum[i], colSum[i] ≤ 10⁸`
- `sum(rowSum) == sum(colSum)`

In [1]:
import random
import time
import heapq
import itertools
from typing import List

examples = [([3, 8], [4, 7]), ([5, 7, 10], [8, 6, 8])]
sample_answers = [[[3, 0], [1, 7]], [[0, 5, 0], [6, 1, 0], [2, 0, 8]]]

def is_valid(mat, rowSum, colSum):
    """Hàm kiểm tra hợp lệ: đúng kích thước, số nguyên không âm, tổng hàng/cột khớp."""
    m, n = len(rowSum), len(colSum)
    if len(mat) != m or any(len(r) != n for r in mat):
        return False
    if any(not isinstance(x, int) or x < 0 for r in mat for x in r):
        return False
    if [sum(r) for r in mat] != list(rowSum):
        return False
    return [sum(mat[i][j] for i in range(m)) for j in range(n)] == list(colSum)

for (r, c), ans in zip(examples, sample_answers):
    print(f"rowSum={r}, colSum={c}, đáp án mẫu {ans} hợp lệ? {is_valid(ans, r, c)}")
print("đáp án khác [[1,2],[3,5]] hợp lệ?", is_valid([[1, 2], [3, 5]], [3, 8], [4, 7]))

rowSum=[3, 8], colSum=[4, 7], đáp án mẫu [[3, 0], [1, 7]] hợp lệ? True
rowSum=[5, 7, 10], colSum=[8, 6, 8], đáp án mẫu [[0, 5, 0], [6, 1, 0], [2, 0, 8]] hợp lệ? True
đáp án khác [[1,2],[3,5]] hợp lệ? True


## 3. Kiến thức nền

### 3.1 Tạo ma trận `m × n` đúng cách

`[[0] * n for _ in range(m)]` — mỗi hàng là một list riêng. `[[0] * n] * m` tạo `m` tham chiếu tới **cùng một** list (lỗi kinh điển — xem mục 13).

In [2]:
m, n = 2, 3
mat = [[0] * n for _ in range(m)]
mat[0][1] = 5
print(mat)

[[0, 5, 0], [0, 0, 0]]


### 3.2 Tổng hàng, tổng cột

Tổng hàng: `sum(row)`. Tổng cột: `sum(mat[i][j] for i in range(m))` hoặc gọn hơn `[sum(col) for col in zip(*mat)]` (`zip(*mat)` "chuyển vị" ma trận).

In [3]:
mat = [[0, 5, 0], [6, 1, 0], [2, 0, 8]]
print("tổng hàng:", [sum(r) for r in mat])
print("tổng cột :", [sum(c) for c in zip(*mat)])

tổng hàng: [5, 7, 10]
tổng cột : [8, 6, 8]


### 3.3 `min` và tư duy "làm cạn"

Đặt `x = min(a, b)` rồi trừ `x` khỏi cả `a` và `b`: sau đó **ít nhất một** trong hai bằng 0, và không cái nào âm.

In [4]:
for a, b in [(3, 4), (8, 7), (5, 5), (0, 9)]:
    x = min(a, b)
    print(f"a={a}, b={b} -> đặt {x} -> còn a={a - x}, b={b - x}")

a=3, b=4 -> đặt 3 -> còn a=0, b=1
a=8, b=7 -> đặt 7 -> còn a=1, b=0
a=5, b=5 -> đặt 5 -> còn a=0, b=0
a=0, b=9 -> đặt 0 -> còn a=0, b=9


### 3.4 `heapq` — lấy phần tử nhỏ nhất

`heapq.heappush(h, item)`, `heapq.heappop(h)` O(log k). Dùng cho phiên bản "luôn chọn tổng còn lại nhỏ nhất" (gợi ý của đề).

In [5]:
h = []
for v in [(8, "col0"), (5, "row0"), (6, "col1")]:
    heapq.heappush(h, v)
print(heapq.heappop(h), heapq.heappop(h))

(5, 'row0') (6, 'col1')


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1 (`rowSum = [3,8]`, `colSum = [4,7]`). Bắt đầu ở ô `(0,0)`:
- Hàng 0 còn cần 3, cột 0 còn cần 4 ⇒ đặt `min(3,4) = 3`. Hàng 0 xong (còn 0), cột 0 còn 1.
- Ô `(0,1)`: hàng 0 còn 0 ⇒ đặt 0.
- Ô `(1,0)`: hàng 1 còn 8, cột 0 còn 1 ⇒ đặt 1. Cột 0 xong, hàng 1 còn 7.
- Ô `(1,1)`: hàng 1 còn 7, cột 1 còn 7 ⇒ đặt 7. Cả hai xong.

Kết quả `[[3,0],[1,7]]` — đúng đáp án mẫu.

**Trực giác:** mỗi ô ta "đổ" vào lượng lớn nhất mà cả hàng lẫn cột còn chịu được. Vì tổng toàn bộ hai phía bằng nhau, khi đổ xong mọi thứ sẽ vừa khít.

In [6]:
r, c = [3, 8], [4, 7]
r, c = r[:], c[:]
mat = [[0] * 2 for _ in range(2)]
for i in range(2):
    for j in range(2):
        x = min(r[i], c[j]); mat[i][j] = x; r[i] -= x; c[j] -= x
        print(f"ô ({i},{j}): đặt {x}, hàng còn {r}, cột còn {c}")
print(mat)

ô (0,0): đặt 3, hàng còn [0, 8], cột còn [1, 7]
ô (0,1): đặt 0, hàng còn [0, 8], cột còn [1, 7]
ô (1,0): đặt 1, hàng còn [0, 7], cột còn [0, 7]
ô (1,1): đặt 7, hàng còn [0, 0], cột còn [0, 0]
[[3, 0], [1, 7]]


## 5. Cách 1 — Brute Force / baseline: quay lui thử giá trị từng ô

**Ý tưởng:** điền ô theo thứ tự hàng–cột. Ô `(i, j)` nhận giá trị `v` từ `0` đến `min(remR[i], remC[j])`. Cắt nhánh: ở **ô cuối của hàng**, giá trị bắt buộc là `remR[i]` (phần còn lại của hàng); ở **hàng cuối**, giá trị bắt buộc là `remC[j]`. Gặp ma trận đầy đủ hợp lệ thì trả về.

Chỉ dùng cho input rất nhỏ — số khả năng bùng nổ theo cả kích thước lẫn độ lớn tổng.

In [7]:
def restore_brute(rowSum, colSum):
    m, n = len(rowSum), len(colSum)
    remR, remC = rowSum[:], colSum[:]
    mat = [[0] * n for _ in range(m)]
    def go(k):
        if k == m * n:
            return all(x == 0 for x in remR) and all(x == 0 for x in remC)
        i, j = divmod(k, n)
        hi = min(remR[i], remC[j])
        if j == n - 1:
            choices = [remR[i]] if remR[i] <= remC[j] else []      # ô cuối hàng: bắt buộc
        elif i == m - 1:
            choices = [remC[j]] if remC[j] <= remR[i] else []      # hàng cuối: bắt buộc
        else:
            choices = range(hi, -1, -1)
        for v in choices:
            mat[i][j] = v; remR[i] -= v; remC[j] -= v
            if go(k + 1):
                return True
            remR[i] += v; remC[j] += v
        mat[i][j] = 0
        return False
    return mat if go(0) else None

for r, c in examples:
    res = restore_brute(r, c)
    print(r, c, "->", res, "hợp lệ:", is_valid(res, r, c))

[3, 8] [4, 7] -> [[3, 0], [1, 7]] hợp lệ: True
[5, 7, 10] [8, 6, 8] -> [[5, 0, 0], [3, 4, 0], [0, 2, 8]] hợp lệ: True


**Độ phức tạp:** worst case hàm mũ — khoảng `∏ (min(rᵢ, cⱼ) + 1)` nhánh. Ở bản trên, thử giá trị **lớn trước** nên thường tìm ra ngay (đó chính là gợi ý cho cách tham lam!). Nếu thử nhỏ trước, số nhánh tăng vọt — đo bằng đếm số lần gọi:

In [8]:
def count_calls(rowSum, colSum, big_first):
    m, n = len(rowSum), len(colSum)
    remR, remC = rowSum[:], colSum[:]
    calls = 0
    def go(k):
        nonlocal calls
        calls += 1
        if k == m * n:
            return all(x == 0 for x in remR) and all(x == 0 for x in remC)
        i, j = divmod(k, n)
        hi = min(remR[i], remC[j])
        rng = range(hi, -1, -1) if big_first else range(hi + 1)
        for v in rng:
            remR[i] -= v; remC[j] -= v
            if go(k + 1):
                return True
            remR[i] += v; remC[j] += v
        return False
    go(0)
    return calls

r, c = [6, 7, 5], [3, 9, 6]
print("thử nhỏ trước:", count_calls(r, c, False), "lần gọi | thử lớn trước:", count_calls(r, c, True), "lần gọi")

thử nhỏ trước: 23537 lần gọi | thử lớn trước: 10 lần gọi


**Nút thắt:** tìm kiếm mù rất tốn; nhưng quan sát "thử lớn trước gần như không cần quay lui" gợi ý rằng **luôn đặt giá trị lớn nhất có thể** là một chiến lược an toàn.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight (tham lam):** tại ô `(i, j)`, đặt `x = min(remR[i], remC[j])` rồi trừ cho cả hai. Khi đó:
- không bao giờ âm (vì `x` không vượt quá cái nào);
- **ít nhất một** trong hai (hàng `i` hoặc cột `j`) **cạn về 0** ⇒ có thể bỏ hẳn hàng/cột đó khỏi bài toán;
- phần còn lại vẫn là một bài toán cùng dạng với `Σ remR == Σ remC` (vì ta trừ cùng lượng `x` ở hai phía).

**Quy tắc hai con trỏ:** bắt đầu `i = j = 0`:
- đặt `x = min(rowSum[i], colSum[j])` vào ô `(i, j)`, trừ hai phía;
- **nếu** hàng `i` cạn **thì** `i += 1`; **nếu** cột `j` cạn **thì** `j += 1` (có thể cả hai);
- dừng khi `i == m` hoặc `j == n`. Các ô chưa chạm tới để 0.

**Invariant:** trước mỗi bước, các hàng `< i` và các cột `< j` đã đủ tổng; `Σ_{i' ≥ i} remR[i'] = Σ_{j' ≥ j} remC[j']`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tham lam duyệt mọi ô (hàng–cột)

Đơn giản nhất: hai vòng `for` lồng nhau, mỗi ô `min` rồi trừ. Ô có hàng hoặc cột đã cạn tự động nhận 0.

In [9]:
def restore_greedy_cells(rowSum, colSum):
    r, c = rowSum[:], colSum[:]          # sao chép để không sửa input
    m, n = len(r), len(c)
    mat = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            x = min(r[i], c[j])
            mat[i][j] = x
            r[i] -= x
            c[j] -= x
    return mat

for r, c in examples:
    res = restore_greedy_cells(r, c)
    print(r, c, "->", res, "hợp lệ:", is_valid(res, r, c))

[3, 8] [4, 7] -> [[3, 0], [1, 7]] hợp lệ: True
[5, 7, 10] [8, 6, 8] -> [[5, 0, 0], [3, 4, 0], [0, 2, 8]] hợp lệ: True


Code nộp LeetCode của Cách 2:

```python
class Solution:
    def restoreMatrix(self, rowSum: List[int], colSum: List[int]) -> List[List[int]]:
        m, n = len(rowSum), len(colSum)
        mat = [[0] * n for _ in range(m)]
        for i in range(m):
            for j in range(n):
                x = min(rowSum[i], colSum[j])
                mat[i][j] = x
                rowSum[i] -= x
                colSum[j] -= x
        return mat
```

**Độ phức tạp:** Time O(m·n), Space O(m·n) cho kết quả (O(1) phụ, nếu được sửa input).

### 7.2 Cách 3 — Hai con trỏ (chỉ chạm O(m + n) ô)

Mỗi bước làm cạn ít nhất một hàng hoặc một cột ⇒ tối đa `m + n − 1` bước điền. Ma trận vẫn phải cấp phát O(m·n) (đầu ra), nhưng số thao tác "thật" ít hơn, và kết quả có **nhiều nhất `m + n − 1` ô khác 0**.

In [10]:
def restore_two_pointers(rowSum, colSum):
    r, c = rowSum[:], colSum[:]
    m, n = len(r), len(c)
    mat = [[0] * n for _ in range(m)]
    i = j = 0
    while i < m and j < n:
        x = min(r[i], c[j])
        mat[i][j] = x
        r[i] -= x
        c[j] -= x
        if r[i] == 0:
            i += 1
        if c[j] == 0:
            j += 1
    return mat

for r, c in examples:
    res = restore_two_pointers(r, c)
    print(r, c, "->", res, "hợp lệ:", is_valid(res, r, c))

[3, 8] [4, 7] -> [[3, 0], [1, 7]] hợp lệ: True
[5, 7, 10] [8, 6, 8] -> [[5, 0, 0], [3, 4, 0], [0, 2, 8]] hợp lệ: True


Khi `r[i] == c[j]`, cả hai con trỏ cùng tiến — đúng, vì cả hàng lẫn cột đều đã đủ. Với hàng/cột có tổng 0 ngay từ đầu, `x = 0` và con trỏ tương ứng tiến qua.

Code nộp LeetCode của Cách 3: xem mục 9.

**Độ phức tạp:** Time O(m·n) cho cấp phát + O(m + n) bước điền; Space O(m·n) cho kết quả.

### 7.3 Cách 4 — Theo gợi ý của đề: luôn xử lý tổng nhỏ nhất (heap)

Gợi ý: "tìm `rowSum` hoặc `colSum` nhỏ nhất, gọi là `x`; đặt `x` vào lưới, trừ khỏi hàng và cột tương ứng". Cụ thể: lấy phần tử nhỏ nhất trong cả hai phía (giả sử là hàng `i` với giá trị `x`); ghép với **bất kỳ** cột `j` còn lại (mọi cột còn lại đều có `remC[j] ≥ x` vì `x` nhỏ nhất) — ta chọn cột nhỏ nhất bên phía cột. Đặt `x` vào `(i, j)`, hàng `i` xong, cột `j` giảm `x`.

Dùng hai heap (một cho hàng, một cho cột). Cách này minh họa rằng tham lam có nhiều "thứ tự" đều đúng.

In [11]:
def restore_heap(rowSum, colSum):
    m, n = len(rowSum), len(colSum)
    mat = [[0] * n for _ in range(m)]
    hr = [(v, i) for i, v in enumerate(rowSum)]
    hc = [(v, j) for j, v in enumerate(colSum)]
    heapq.heapify(hr); heapq.heapify(hc)
    while hr and hc:
        (rv, i), (cv, j) = hr[0], hc[0]
        x = min(rv, cv)
        mat[i][j] = x
        heapq.heappop(hr); heapq.heappop(hc)
        if rv - x > 0:
            heapq.heappush(hr, (rv - x, i))
        if cv - x > 0:
            heapq.heappush(hc, (cv - x, j))
    return mat

for r, c in examples:
    res = restore_heap(r, c)
    print(r, c, "->", res, "hợp lệ:", is_valid(res, r, c))

[3, 8] [4, 7] -> [[3, 0], [1, 7]] hợp lệ: True
[5, 7, 10] [8, 6, 8] -> [[0, 5, 0], [6, 1, 0], [2, 0, 8]] hợp lệ: True


Lưu ý: ở mỗi bước ta pop cả hai đỉnh và chỉ đẩy lại phía còn dư (phía bằng `x` đã cạn). Nếu cả hai bằng nhau thì cả hai cùng cạn. Vì mỗi cặp `(i, j)` chỉ được ghép một lần (sau đó ít nhất một bên biến mất), không ô nào bị ghi đè.

Code nộp LeetCode của Cách 4:

```python
import heapq

class Solution:
    def restoreMatrix(self, rowSum: List[int], colSum: List[int]) -> List[List[int]]:
        m, n = len(rowSum), len(colSum)
        mat = [[0] * n for _ in range(m)]
        hr = [(v, i) for i, v in enumerate(rowSum)]
        hc = [(v, j) for j, v in enumerate(colSum)]
        heapq.heapify(hr); heapq.heapify(hc)
        while hr and hc:
            (rv, i), (cv, j) = heapq.heappop(hr), heapq.heappop(hc)
            x = min(rv, cv)
            mat[i][j] = x
            if rv > x:
                heapq.heappush(hr, (rv - x, i))
            if cv > x:
                heapq.heappush(hc, (cv - x, j))
        return mat
```

**Độ phức tạp:** Time O(m·n + (m + n) log(m + n)), Space O(m·n).

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 2 (`rowSum = [5,7,10]`, `colSum = [8,6,8]`):

| bước | (i, j) | r[i] | c[j] | đặt x | r sau | c sau | tiến |
|---|---|---|---|---|---|---|---|
| 1 | (0,0) | 5 | 8 | 5 | [0,7,10] | [3,6,8] | i |
| 2 | (1,0) | 7 | 3 | 3 | [0,4,10] | [0,6,8] | j |
| 3 | (1,1) | 4 | 6 | 4 | [0,0,10] | [0,2,8] | i |
| 4 | (2,1) | 10 | 2 | 2 | [0,0,8] | [0,0,8] | j |
| 5 | (2,2) | 8 | 8 | 8 | [0,0,0] | [0,0,0] | i, j |

Kết quả `[[5,0,0],[3,4,0],[0,2,8]]` — khác đáp án mẫu nhưng **cũng hợp lệ**.

In [12]:
def trace_two_pointers(rowSum, colSum):
    r, c = rowSum[:], colSum[:]
    m, n = len(r), len(c)
    mat = [[0] * n for _ in range(m)]
    i = j = step = 0
    while i < m and j < n:
        step += 1
        x = min(r[i], c[j]); mat[i][j] = x
        r[i] -= x; c[j] -= x
        moves = []
        old = (i, j)
        if r[i] == 0: i += 1; moves.append("i")
        if c[j] == 0: j += 1; moves.append("j")
        print(f"bước {step}: ô {old} đặt {x:3d} | r={r} c={c} | tiến {','.join(moves)}")
    for row in mat:
        print("   ", row)
    print("hợp lệ:", is_valid(mat, rowSum, colSum))
    return mat

trace_two_pointers([5, 7, 10], [8, 6, 8])

bước 1: ô (0, 0) đặt   5 | r=[0, 7, 10] c=[3, 6, 8] | tiến i
bước 2: ô (1, 0) đặt   3 | r=[0, 4, 10] c=[0, 6, 8] | tiến j
bước 3: ô (1, 1) đặt   4 | r=[0, 0, 10] c=[0, 2, 8] | tiến i
bước 4: ô (2, 1) đặt   2 | r=[0, 0, 8] c=[0, 0, 8] | tiến j
bước 5: ô (2, 2) đặt   8 | r=[0, 0, 0] c=[0, 0, 0] | tiến i,j
    [5, 0, 0]
    [3, 4, 0]
    [0, 2, 8]
hợp lệ: True


[[5, 0, 0], [3, 4, 0], [0, 2, 8]]

In [13]:
# Trace có hàng/cột tổng 0 và trường hợp hai phía bằng nhau
trace_two_pointers([0, 4, 0, 6], [4, 0, 6])

bước 1: ô (0, 0) đặt   0 | r=[0, 4, 0, 6] c=[4, 0, 6] | tiến i
bước 2: ô (1, 0) đặt   4 | r=[0, 0, 0, 6] c=[0, 0, 6] | tiến i,j
bước 3: ô (2, 1) đặt   0 | r=[0, 0, 0, 6] c=[0, 0, 6] | tiến i,j
bước 4: ô (3, 2) đặt   6 | r=[0, 0, 0, 0] c=[0, 0, 0] | tiến i,j
    [0, 0, 0]
    [4, 0, 0]
    [0, 0, 0]
    [0, 0, 6]
hợp lệ: True


[[0, 0, 0], [4, 0, 0], [0, 0, 0], [0, 0, 6]]

## 9. Code nộp LeetCode

```python
from typing import List

class Solution:
    def restoreMatrix(self, rowSum: List[int], colSum: List[int]) -> List[List[int]]:
        m, n = len(rowSum), len(colSum)
        mat = [[0] * n for _ in range(m)]
        i = j = 0
        while i < m and j < n:
            x = min(rowSum[i], colSum[j])   # đổ nhiều nhất có thể vào ô (i, j)
            mat[i][j] = x
            rowSum[i] -= x
            colSum[j] -= x
            if rowSum[i] == 0:
                i += 1                      # hàng i đã đủ
            if colSum[j] == 0:
                j += 1                      # cột j đã đủ
        return mat
```

In [14]:
class Solution:
    def restoreMatrix(self, rowSum: List[int], colSum: List[int]) -> List[List[int]]:
        m, n = len(rowSum), len(colSum)
        mat = [[0] * n for _ in range(m)]
        i = j = 0
        while i < m and j < n:
            x = min(rowSum[i], colSum[j])
            mat[i][j] = x
            rowSum[i] -= x
            colSum[j] -= x
            if rowSum[i] == 0:
                i += 1
            if colSum[j] == 0:
                j += 1
        return mat

for r, c in examples:
    res = Solution().restoreMatrix(r[:], c[:])       # truyền bản sao vì hàm sửa input
    print(r, c, "->", res, "hợp lệ:", is_valid(res, r, c))

[3, 8] [4, 7] -> [[3, 0], [1, 7]] hợp lệ: True
[5, 7, 10] [8, 6, 8] -> [[5, 0, 0], [3, 4, 0], [0, 2, 8]] hợp lệ: True


## 10. Vì sao thuật toán đúng

**Bổ đề:** nếu `Σ r = Σ c` (mọi phần tử không âm), thì sau khi đặt `x = min(r[i], c[j])` vào ô `(i, j)` và trừ hai phía, bài toán còn lại (bỏ hàng/cột đã cạn) vẫn có `Σ r' = Σ c'` với phần tử không âm.

**Chứng minh hai con trỏ đúng:**
1. Mọi giá trị đặt vào là `min` của hai số không âm ⇒ không âm, nguyên.
2. Không hàng/cột nào vượt tổng: ta chỉ trừ tới 0, không quá.
3. **Kết thúc với mọi tổng đều đủ:** vòng lặp dừng khi `i == m` hoặc `j == n`. Giả sử dừng vì `i == m`: mọi hàng đã cạn ⇒ tổng còn lại phía hàng = 0 ⇒ theo invariant tổng còn lại phía cột cũng = 0 ⇒ mọi cột cũng đủ (vì phần tử không âm, tổng 0 nghĩa là từng cột bằng 0). Trường hợp `j == n` đối xứng.
4. Mỗi bước, ít nhất một con trỏ tiến ⇒ tối đa `m + n` bước ⇒ kết thúc.

Điều kiện đề "luôn tồn tại" chính là `Σ rowSum = Σ colSum` — và chứng minh trên cho thấy điều kiện này **đủ**: tham lam luôn dựng được ma trận.

## 11. Phân tích Time / Space

Gọi `m = len(rowSum)`, `n = len(colSum)` (≤ 500).

| Cách | Time | Space (ngoài đầu ra) |
|---|---|---|
| 1. Quay lui | hàm mũ | O(m·n) đệ quy |
| 2. Tham lam mọi ô | O(m·n) | O(1) (hoặc O(m+n) nếu sao chép input) |
| 3. Hai con trỏ | O(m·n) cấp phát + O(m+n) điền | O(1) |
| 4. Heap | O(m·n + (m+n) log(m+n)) | O(m+n) |

Đầu ra đã là `m·n` số nên không thể nhanh hơn O(m·n). Đo với `500 × 500`:

In [15]:
def random_sums(m, n, rnd, hi=10**8):
    """Sinh rowSum, colSum hợp lệ bằng cách sinh ma trận ẩn ngẫu nhiên (đảm bảo tồn tại)."""
    r, c = [0] * m, [0] * n
    for i in range(m):
        for j in range(n):
            if rnd.random() < 0.3:
                v = rnd.randint(0, hi // max(m, n)); r[i] += v; c[j] += v
    return r, c

rnd = random.Random(0)
R, Cc = random_sums(500, 500, rnd)
for f in [restore_greedy_cells, restore_two_pointers, restore_heap]:
    t0 = time.perf_counter(); res = f(R, Cc)
    nz = sum(1 for row in res for x in row if x)
    print(f"{f.__name__:22s} hợp lệ={is_valid(res, R, Cc)}  ô khác 0={nz:6d}  {time.perf_counter() - t0:.3f}s")

restore_greedy_cells   hợp lệ=True  ô khác 0=   999  0.072s


restore_two_pointers   hợp lệ=True  ô khác 0=   999  0.039s


restore_heap           hợp lệ=True  ô khác 0=   999  0.036s


Cách 3 cho tối đa `m + n − 1 = 999` ô khác 0 (ma trận "thưa"), trong khi Cách 2 cũng vậy (thực chất Cách 2 và Cách 3 cho **cùng** ma trận — Cách 2 chỉ tốn thêm thời gian duyệt các ô chắc chắn bằng 0).

## 12. Bộ test

Vì nhiều đáp án, mọi test dùng `is_valid`. Gồm: ví dụ đề, toàn 0, 1×1, 1×n, m×1, hàng/cột có tổng 0 xen kẽ, giá trị lớn `10⁸`, và input ngẫu nhiên sinh từ ma trận ẩn (bảo đảm tồn tại lời giải). Đồng thời kiểm tra hàm **không** làm hỏng input (với các cách sao chép) và đối chiếu với quay lui ở kích thước nhỏ.

In [16]:
ALL = [restore_greedy_cells, restore_two_pointers, restore_heap,
       lambda r, c: Solution().restoreMatrix(r[:], c[:])]
fixed = examples + [
    ([0], [0]), ([7], [7]), ([5], [1, 2, 2]), ([1, 2, 2], [5]),
    ([0, 0], [0, 0, 0]), ([0, 3, 0], [1, 0, 2]), ([10**8, 10**8], [10**8, 10**8]),
    ([4, 0, 6], [0, 10]), ([1, 1, 1, 1], [2, 2]),
]
for r, c in fixed:
    for f in ALL:
        r0, c0 = r[:], c[:]
        res = f(r, c)
        assert is_valid(res, r0, c0), (f, r, c, res)
        assert r == r0 and c == c0, "input bị sửa"
    assert is_valid(restore_brute(r, c) if sum(r) <= 12 else restore_two_pointers(r, c), r, c)
print("Ca cố định:", len(fixed), "OK")

rnd = random.Random(2024)
for _ in range(600):
    m, n = rnd.randint(1, 6), rnd.randint(1, 6)
    r, c = random_sums(m, n, rnd, hi=60)
    for f in ALL:
        assert is_valid(f(r, c), r, c)
    if sum(r) <= 15 and m * n <= 9:
        assert is_valid(restore_brute(r, c), r, c)
print("Ngẫu nhiên nhỏ: OK")
for _ in range(20):
    m, n = rnd.randint(1, 200), rnd.randint(1, 200)
    r, c = random_sums(m, n, rnd)
    for f in ALL:
        assert is_valid(f(r, c), r, c)
print("Ngẫu nhiên lớn: OK")

Ca cố định: 11 OK
Ngẫu nhiên nhỏ: OK


Ngẫu nhiên lớn: OK


In [17]:
# Kiểm tra tính chất: hai con trỏ cho tối đa m + n - 1 ô khác 0
rnd = random.Random(7)
for _ in range(500):
    m, n = rnd.randint(1, 8), rnd.randint(1, 8)
    r, c = random_sums(m, n, rnd, hi=100)
    res = restore_two_pointers(r, c)
    assert sum(1 for row in res for x in row if x) <= m + n - 1
print("Số ô khác 0 luôn <= m + n - 1")

Số ô khác 0 luôn <= m + n - 1


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chia theo tỉ lệ `rowSum[i] * colSum[j] / total`

In [18]:
def wrong_proportional(rowSum, colSum):
    total = sum(rowSum)
    return [[round(ri * cj / total) for cj in colSum] for ri in rowSum]   # SAI: làm tròn phá tổng

for r, c in [([3, 8], [4, 7]), ([1, 1, 1], [1, 1, 1]), ([5, 7, 10], [8, 6, 8])]:
    res = wrong_proportional(r, c)
    print(f"{r} {c}: {res} | tổng hàng {[sum(x) for x in res]} | tổng cột {[sum(x) for x in zip(*res)]} | hợp lệ? {is_valid(res, r, c)}")

[3, 8] [4, 7]: [[1, 2], [3, 5]] | tổng hàng [3, 8] | tổng cột [4, 7] | hợp lệ? True
[1, 1, 1] [1, 1, 1]: [[0, 0, 0], [0, 0, 0], [0, 0, 0]] | tổng hàng [0, 0, 0] | tổng cột [0, 0, 0] | hợp lệ? False
[5, 7, 10] [8, 6, 8]: [[2, 1, 2], [3, 2, 3], [4, 3, 4]] | tổng hàng [5, 8, 11] | tổng cột [9, 6, 9] | hợp lệ? False


**Vì sao sai:** với ví dụ 1 nó **tình cờ** đúng (`[[1,2],[3,5]]`) — đó là lý do lỗi này nguy hiểm. Công thức tỉ lệ chỉ đúng tổng **trên số thực**; đề cần **số nguyên**, và làm tròn làm lệch tổng: `[1,1,1]`/`[1,1,1]` cho toàn số 0 (mỗi ô `1/3` làm tròn thành 0). Ngoài ra chia cho 0 khi `total = 0`. **Sửa:** tham lam `min`.

### Lỗi 2 — Quên trừ ở một phía

In [19]:
def wrong_one_side(rowSum, colSum):
    r, c = rowSum[:], colSum[:]
    m, n = len(r), len(c)
    mat = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            x = min(r[i], c[j])
            mat[i][j] = x
            r[i] -= x                                  # SAI: quên c[j] -= x
    return mat

r, c = [3, 8], [4, 7]
res = wrong_one_side(r, c)
print("ma trận:", res, "| tổng cột:", [sum(x) for x in zip(*res)], "(cần", c, ")| hợp lệ?", is_valid(res, r, c))

ma trận: [[3, 0], [4, 4]] | tổng cột: [7, 4] (cần [4, 7] )| hợp lệ? False


**Vì sao sai:** cột 0 không biết mình đã nhận 3 nên tiếp tục nhận `min(8, 4) = 4` ở hàng sau ⇒ tổng cột vượt. **Sửa:** trừ `x` khỏi **cả** hàng và cột.

### Lỗi 3 — Dùng `max` thay vì `min`

In [20]:
def wrong_max(rowSum, colSum):
    r, c = rowSum[:], colSum[:]
    m, n = len(r), len(c)
    mat = [[0] * n for _ in range(m)]
    i = j = 0
    steps = 0
    while i < m and j < n and steps < 20:
        steps += 1
        x = max(r[i], c[j])                            # SAI
        mat[i][j] = x; r[i] -= x; c[j] -= x
        if r[i] <= 0: i += 1
        if c[j] <= 0: j += 1
    return mat

r, c = [3, 8], [4, 7]
res = wrong_max(r, c)
print("ma trận:", res, "| hợp lệ?", is_valid(res, r, c))

ma trận: [[4, 0], [0, 8]] | hợp lệ? False


**Vì sao sai:** đặt 4 vào ô `(0,0)` làm hàng 0 vượt tổng (3 − 4 = −1): không còn cách nào sửa vì phần tử không được âm. **Sửa:** `min` — không bao giờ vượt bên nào.

### Lỗi 4 — Tạo ma trận bằng `[[0] * n] * m`

In [21]:
def wrong_alias(rowSum, colSum):
    r, c = rowSum[:], colSum[:]
    m, n = len(r), len(c)
    mat = [[0] * n] * m                                # SAI: m tham chiếu tới cùng 1 hàng
    for i in range(m):
        for j in range(n):
            x = min(r[i], c[j]); mat[i][j] = x; r[i] -= x; c[j] -= x
    return mat

r, c = [3, 8], [4, 7]
res = wrong_alias(r, c)
print("ma trận:", res, "| hợp lệ?", is_valid(res, r, c))

ma trận: [[1, 7], [1, 7]] | hợp lệ? False


**Vì sao sai:** mọi hàng là **cùng một list**, ghi vào hàng 1 cũng ghi đè hàng 0 ⇒ ma trận cuối cùng có các hàng giống hệt nhau. **Sửa:** `[[0] * n for _ in range(m)]`.

### Bẫy suy nghĩ (không phải lỗi) — Hai con trỏ dùng `elif`

In [22]:
def variant_elif(rowSum, colSum):
    r, c = rowSum[:], colSum[:]
    m, n = len(r), len(c)
    mat = [[0] * n for _ in range(m)]
    i = j = 0
    while i < m and j < n:
        x = min(r[i], c[j]); mat[i][j] = x; r[i] -= x; c[j] -= x
        if r[i] == 0:
            i += 1
        elif c[j] == 0:                                # 'elif' — vẫn đúng! xem giải thích
            j += 1
    return mat

rnd = random.Random(1)
all_ok = True
for _ in range(300):
    rr, cc = random_sums(rnd.randint(1, 6), rnd.randint(1, 6), rnd, 50)
    all_ok &= is_valid(variant_elif(rr, cc), rr, cc)
print("elif hợp lệ trên 300 input ngẫu nhiên:", all_ok)
r, c = [5, 7, 10], [8, 6, 8]
print("elif cho:", variant_elif(r, c), "hợp lệ?", is_valid(variant_elif(r, c), r, c))

elif hợp lệ trên 300 input ngẫu nhiên: True
elif cho: [[5, 0, 0], [3, 4, 0], [0, 2, 8]] hợp lệ? True


**Không phải lỗi!** Đây là một "lỗi tưởng tượng" đáng nhắc: dùng `elif` thì khi cả hai cùng cạn chỉ `i` tiến; bước sau ô `(i+1, j)` nhận `min(r, 0) = 0` rồi `j` tiến. Kết quả vẫn hợp lệ, chỉ thêm vài bước đặt 0. Bài học: **kiểm chứng bằng validator** trước khi kết luận một biến thể là sai.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Return **any** matrix/array satisfying sums" ⇒ thường là **tham lam** "đổ nhiều nhất có thể" (`min`), không cần tìm kiếm.
- Hai danh sách "cung" và "cầu" có tổng bằng nhau ⇒ hai con trỏ ghép cung–cầu (giống **phương pháp góc Tây Bắc** trong bài toán vận tải).
- Bài nhiều đáp án ⇒ viết `is_valid` trước, test mọi cách bằng nó.
- Tag "Flow Network": về lý thuyết là luồng cực đại (nguồn → hàng → cột → đích), nhưng đồ thị **đầy đủ, không giới hạn sức chứa ô** nên tham lam đủ.
- Không sửa input nếu đề/bài phỏng vấn không cho phép — sao chép `r, c = rowSum[:], colSum[:]`.

## 15. Biến thể của bài

1. **Giới hạn mỗi ô** `0 ≤ mat[i][j] ≤ cap[i][j]`: tham lam không còn đúng ⇒ cần **luồng cực đại** (max-flow).
2. **Ma trận nhị phân** (mỗi ô 0/1) với tổng hàng/cột cho trước: định lý **Gale–Ryser**; tham lam "hàng tổng lớn nhất ghép các cột còn nhiều nhất".
3. **LC 2245**, **LC 1253 — Reconstruct a 2-Row Binary Matrix**: phiên bản 2 hàng, mỗi ô 0/1.
4. **Đếm số ma trận** thỏa mãn (bài tập 5) — tổ hợp / DP.
5. **Bài toán vận tải (transportation problem):** thêm chi phí mỗi ô, cực tiểu tổng chi phí — "góc Tây Bắc" chỉ là nghiệm khởi đầu.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space phụ | Khi nào dùng |
|---|---|---|---|---|
| 1. Quay lui | thử mọi giá trị từng ô | hàm mũ | O(m·n) | hiểu bài, input tí hon |
| 2. Tham lam mọi ô | `min` rồi trừ, duyệt hết ô | O(m·n) | O(1) | code ngắn nhất |
| 3. Hai con trỏ | ghép hàng–cột, tiến bên cạn | O(m·n) cấp phát, O(m+n) điền | O(1) | **nộp bài**, ma trận thưa |
| 4. Heap (gợi ý đề) | luôn xử lý tổng nhỏ nhất | O(m·n + (m+n)log) | O(m+n) | minh họa tính linh hoạt của tham lam |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** Kết quả của Cách 3 (hai con trỏ) với `rowSum = [2,3]`, `colSum = [1,4]`? Với `rowSum = [6]`, `colSum = [1,2,3]`?

**Bài 2 (tính tay):** Trace Cách 3 với `rowSum = [4,1,5]`, `colSum = [2,8]` — mỗi bước ghi ô, giá trị đặt và con trỏ nào tiến.

**Bài 3 (điền TODO):** Hoàn thiện `check_matrix(mat, rowSum, colSum)` — trả `True/False` như `is_valid`.

**Bài 4 (tự viết lại):** Viết `restore_from_bottom_right(rowSum, colSum)` chạy hai con trỏ **từ góc dưới phải** (i = m−1, j = n−1, lùi dần) — vẫn phải hợp lệ.

**Bài 5 (mini challenge):** Viết `count_matrices(rowSum, colSum)` đếm **số ma trận** số nguyên không âm thỏa mãn (input nhỏ), đối chiếu với vét cạn `itertools.product`.

In [23]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def check_matrix(mat, rowSum, colSum):
    # TODO: kiểm tra số hàng == len(rowSum), mỗi hàng dài len(colSum)
    # TODO: mọi phần tử là int và >= 0
    # TODO: tổng từng hàng và từng cột khớp
    pass

In [24]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def restore_from_bottom_right(rowSum, colSum):
    # TODO
    pass

def count_matrices(rowSum, colSum):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: ô `(0,0)` nhận `min(2,1) = 1`. Mức 2: cột 0 cạn ⇒ `j = 1`; ô `(0,1)` nhận 1; hàng 0 cạn; ô `(1,1)` nhận 3. Mức 3: `[[1,1],[0,3]]`; ví dụ thứ hai `[[1,2,3]]`.

**Bài 2.** Mức 1: `(0,0)` đặt 2, cột 0 cạn. Mức 2: `(0,1)` đặt 2, hàng 0 cạn; `(1,1)` đặt 1. Mức 3: `(2,1)` đặt 5 ⇒ `[[2,2],[0,1],[0,5]]`.

**Bài 3.** Mức 1: giống `is_valid`. Mức 2: `isinstance(x, int)`. Mức 3: tổng cột dùng `zip(*mat)`.

**Bài 4.** Mức 1: đổi chiều chỉ số. Mức 2: `while i >= 0 and j >= 0`. Mức 3: cạn thì `i -= 1` / `j -= 1`.

**Bài 5.** Mức 1: quay lui như Cách 1 nhưng **đếm** thay vì dừng. Mức 2: ô cuối hàng/hàng cuối bị ép giá trị. Mức 3: vét cạn: mỗi ô từ 0 tới `min(rᵢ, cⱼ)`, `itertools.product` rồi lọc bằng `is_valid`.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2
b1 = [restore_two_pointers([2, 3], [1, 4]), restore_two_pointers([6], [1, 2, 3])]
print("Bài 1:", b1)
assert b1 == [[[1, 1], [0, 3]], [[1, 2, 3]]]
assert trace_two_pointers([4, 1, 5], [2, 8]) == [[2, 2], [0, 1], [0, 5]]

Bài 1: [[[1, 1], [0, 3]], [[1, 2, 3]]]
bước 1: ô (0, 0) đặt   2 | r=[2, 1, 5] c=[0, 8] | tiến j
bước 2: ô (0, 1) đặt   2 | r=[0, 1, 5] c=[0, 6] | tiến i
bước 3: ô (1, 1) đặt   1 | r=[0, 0, 5] c=[0, 5] | tiến i
bước 4: ô (2, 1) đặt   5 | r=[0, 0, 0] c=[0, 0] | tiến i,j
    [2, 2]
    [0, 1]
    [0, 5]
hợp lệ: True


In [26]:
# Bài 3
def check_matrix_sol(mat, rowSum, colSum):
    if len(mat) != len(rowSum) or any(len(row) != len(colSum) for row in mat):
        return False
    if any(not isinstance(x, int) or x < 0 for row in mat for x in row):
        return False
    return [sum(r) for r in mat] == list(rowSum) and [sum(c) for c in zip(*mat)] == list(colSum)

assert check_matrix_sol([[3, 0], [1, 7]], [3, 8], [4, 7])
assert not check_matrix_sol([[3, 0], [1, 6]], [3, 8], [4, 7])
assert not check_matrix_sol([[4, -1], [0, 8]], [3, 8], [4, 7])
assert not check_matrix_sol([[1.5, 1.5], [2.5, 5.5]], [3, 8], [4, 7])
rnd = random.Random(3)
for _ in range(300):
    r, c = random_sums(rnd.randint(1, 5), rnd.randint(1, 5), rnd, 40)
    res = restore_two_pointers(r, c)
    assert check_matrix_sol(res, r, c) == is_valid(res, r, c) == True
print("Bài 3: OK")

Bài 3: OK


In [27]:
# Bài 4
def restore_from_bottom_right_sol(rowSum, colSum):
    r, c = rowSum[:], colSum[:]
    m, n = len(r), len(c)
    mat = [[0] * n for _ in range(m)]
    i, j = m - 1, n - 1
    while i >= 0 and j >= 0:
        x = min(r[i], c[j]); mat[i][j] = x; r[i] -= x; c[j] -= x
        if r[i] == 0: i -= 1
        if c[j] == 0: j -= 1
    return mat

rnd = random.Random(4)
for _ in range(500):
    r, c = random_sums(rnd.randint(1, 7), rnd.randint(1, 7), rnd, 80)
    assert is_valid(restore_from_bottom_right_sol(r, c), r, c)
print("Bài 4: OK;", restore_from_bottom_right_sol([3, 8], [4, 7]))

Bài 4: OK; [[3, 0], [1, 7]]


In [28]:
# Bài 5
def count_matrices_sol(rowSum, colSum):
    m, n = len(rowSum), len(colSum)
    remR, remC = rowSum[:], colSum[:]
    def go(k):
        if k == m * n:
            return 1 if all(x == 0 for x in remR) and all(x == 0 for x in remC) else 0
        i, j = divmod(k, n)
        if j == n - 1:
            vals = [remR[i]] if remR[i] <= remC[j] else []
        elif i == m - 1:
            vals = [remC[j]] if remC[j] <= remR[i] else []
        else:
            vals = range(min(remR[i], remC[j]) + 1)
        total = 0
        for v in vals:
            remR[i] -= v; remC[j] -= v
            total += go(k + 1)
            remR[i] += v; remC[j] += v
        return total
    return go(0)

def count_matrices_brute(rowSum, colSum):
    m, n = len(rowSum), len(colSum)
    ranges = [range(min(rowSum[i], colSum[j]) + 1) for i in range(m) for j in range(n)]
    cnt = 0
    for flat in itertools.product(*ranges):
        mat = [list(flat[i * n:(i + 1) * n]) for i in range(m)]
        cnt += is_valid(mat, rowSum, colSum)
    return cnt

print("Bài 5: số ma trận cho ví dụ 1 =", count_matrices_sol([3, 8], [4, 7]))
rnd = random.Random(5)
for _ in range(80):
    m, n = rnd.randint(1, 3), rnd.randint(1, 3)
    r, c = random_sums(m, n, rnd, 12)
    if sum(r) <= 8:
        assert count_matrices_sol(r, c) == count_matrices_brute(r, c)
print("Bài 5: OK")

Bài 5: số ma trận cho ví dụ 1 = 4
Bài 5: OK


Ví dụ 1 có đúng 4 ma trận hợp lệ (ô `(0,0)` nhận 0..3, mọi ô khác bị ép) — đó là lý do đề nói "any matrix".

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template tham lam hai con trỏ (cung – cầu):**

```python
mat = [[0] * n for _ in range(m)]
i = j = 0
while i < m and j < n:
    x = min(rowSum[i], colSum[j])
    mat[i][j] = x
    rowSum[i] -= x; colSum[j] -= x
    if rowSum[i] == 0: i += 1
    if colSum[j] == 0: j += 1
return mat
```

**Checklist:**
- [ ] Dùng `min` (không vượt tổng bên nào)?
- [ ] Trừ ở **cả** hàng lẫn cột?
- [ ] Ma trận tạo bằng list comprehension (không alias hàng)?
- [ ] Xử lý hàng/cột tổng 0, 1×1, 1×n, m×1?
- [ ] Test bằng **validator** (nhiều đáp án đúng), không so khớp với đáp án mẫu?
- [ ] Có được phép sửa input không? Nếu không, sao chép trước.